In [1]:
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from dotenv import load_dotenv

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch import nn
from transformers import (
    Trainer, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
)

load_dotenv()

True

In [2]:
df = pd.read_csv("../data/dontpatronizeme_pcl.tsv", sep="\t")
df["label_pcl"] = (df["label"] >= 2).astype(int)
len(df)

10469

In [3]:
# Custom stop words based on EDA
custom_stopwords = {'said', 'the', 'is', 'and'}


def clean_and_prepend(row):
    # Filter stop words from the text
    words = str(row['text']).lower().split()
    cleaned_words = [word for word in words if word not in custom_stopwords]
    cleaned_text = " ".join(cleaned_words)

    # Glue the keyword and country to the front
    return f"Keyword: {row['keyword']}. Country: {row['country_code']}. Text: {cleaned_text}"


# Apply the function across the rows (axis=1)
df["cleaned_text"] = df.apply(clean_and_prepend, axis=1)
df.head()

,par_id,art_id,keyword,country_code,text,label,label_pcl,cleaned_text
0,1,@@24942188,hopeless,ph,"We 're living in times of absolute insanity , ...",0,0,Keyword: hopeless. Country: ph. Text: we 're l...
1,2,@@21968160,migrant,gh,"In Libya today , there are countless number of...",0,0,Keyword: migrant. Country: gh. Text: in libya ...
2,3,@@16584954,immigrant,ie,White House press secretary Sean Spicer said t...,0,0,Keyword: immigrant. Country: ie. Text: white h...
3,4,@@7811231,disabled,nz,Council customers only signs would be displaye...,0,0,Keyword: disabled. Country: nz. Text: council ...
4,5,@@1494111,refugee,ca,""" Just like we received migrants fleeing El Sa...",0,0,"Keyword: refugee. Country: ca. Text: "" just li..."


In [4]:
df_train_dev_labels = pd.read_csv("../data/train_semeval_parids-labels.csv")
df_test_labels = pd.read_csv("../data/dev_semeval_parids-labels.csv")
(
    len(df_train_dev_labels),
    len(df_test_labels),
    len(df_train_dev_labels) + len(df_test_labels)
)

(8375, 2094, 10469)

In [5]:
df_train_dev = df[df["par_id"].isin(df_train_dev_labels["par_id"])]
len(df_train_dev)

8375

In [6]:
df_test = df[df["par_id"].isin(df_test_labels["par_id"])]
len(df_test)

2094

In [7]:
df_train, df_dev = train_test_split(
    df_train_dev,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=df_train_dev["label_pcl"],
)
len(df_train), len(df_dev)

(6700, 1675)

In [8]:
df_train["label_pcl"].value_counts(normalize=True)

label_pcl
0    0.905224
1    0.094776
Name: proportion, dtype: float64

In [9]:
df_dev["label_pcl"].value_counts(normalize=True)

label_pcl
0    0.905075
1    0.094925
Name: proportion, dtype: float64

In [10]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")


def tokenize_function(examples):
    return tokenizer(examples["cleaned_text"], padding="max_length", truncation=True, max_length=128)

In [11]:
hf_dataset_train = Dataset.from_pandas(df_train)
tokenized_dataset_train = hf_dataset_train.map(tokenize_function, batched=True)
tokenized_dataset_train = tokenized_dataset_train.rename_column("label_pcl", "labels")
tokenized_dataset_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/6700 [00:00<?, ? examples/s]

In [12]:
hf_dataset_dev = Dataset.from_pandas(df_dev)
tokenized_dataset_dev = hf_dataset_dev.map(tokenize_function, batched=True)
tokenized_dataset_dev = tokenized_dataset_dev.rename_column("label_pcl", "labels")
tokenized_dataset_dev.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/1675 [00:00<?, ? examples/s]

In [13]:
# Check for Apple Silicon (mps), then NVIDIA (cuda), then fall back to CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
device

device(type='mps')

In [14]:
class_weights = compute_class_weight(
    "balanced",
    classes=np.unique(df_train["label_pcl"]),
    y=df_train["label_pcl"]
)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)


# 2. Create a Custom Trainer to use the weights
class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        # Forward pass
        outputs = model(**inputs)
        logits = outputs.logits
        # Apply the custom weights to the Cross Entropy Loss
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

In [29]:
model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

training_args = TrainingArguments(
    output_dir=".",
    eval_strategy="epoch",
    num_train_epochs=15,
    per_device_train_batch_size=32,
)

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset_train,
    eval_dataset=tokenized_dataset_dev,
)

trainer.train(resume_from_checkpoint=True)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddi

Epoch,Training Loss,Validation Loss
11,0.666379,0.547487
12,0.605874,0.578339
13,0.605874,0.516792
14,0.605874,0.461599
15,0.484685,0.477595


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/henrytsyu/Imperial/70016-cw/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/henrytsyu/Imperial/70016-cw/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=3150, training_loss=0.1744532485235305, metrics={'train_runtime': 576.3849, 'train_samples_per_second': 174.363, 'train_steps_per_second': 5.465, 'total_flos': 5293531322787840.0, 'train_loss': 0.1744532485235305, 'epoch': 15.0})

In [30]:
from sklearn.metrics import f1_score

# 1. Get the model's predictions on your internal dev set
predictions = trainer.predict(tokenized_dataset_dev)

# 2. Convert raw logits into 0 or 1 labels
preds = np.argmax(predictions.predictions, axis=-1)

# 3. Get the true labels
labels = predictions.label_ids

# 4. Calculate the F1 score for the positive class (PCL = 1)
f1 = f1_score(labels, preds)
print("Internal Dev F1 Score:", f1)

/Users/henrytsyu/Imperial/70016-cw/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Internal Dev F1 Score: 0.4398496240601504


In [31]:
from sklearn.metrics import classification_report

# This will show you exactly how many False Positives vs False Negatives you have
print(classification_report(labels, preds))

              precision    recall  f1-score   support

           0       0.97      0.83      0.89      1516
           1       0.31      0.74      0.44       159

    accuracy                           0.82      1675
   macro avg       0.64      0.78      0.67      1675
weighted avg       0.91      0.82      0.85      1675



In [32]:
trainer.save_model("./BestModel")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [28]:
tokenizer.save_pretrained("./BestModel")

('./BestModel/tokenizer_config.json', './BestModel/tokenizer.json')